# 💳 Credit Card Fraud Detection - Exploratory Analysis

This notebook explores the credit card fraud detection dataset and demonstrates the model training pipeline.

**Contents:**
1. Data Loading and Overview
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Model Training and Evaluation
5. Prediction Examples

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

## 1️⃣ Data Loading and Overview

In [ ]:
# Load dataset
try:
    df = pd.read_csv('../data/creditcard.csv')
    print("✅ Dataset loaded successfully!")
except FileNotFoundError:
    print("⚠️ Dataset not found. Generating synthetic data...")
    
    # Generate synthetic data
    np.random.seed(42)
    n_samples = 100000
    n_fraud = int(n_samples * 0.0017)
    n_normal = n_samples - n_fraud
    
    # Normal transactions
    normal_v = np.random.randn(n_normal, 28)
    
    # Fraudulent transactions
    fraud_v = np.random.randn(n_fraud, 28)
    fraud_shift = np.random.uniform(-3, 3, 28)
    fraud_scale = np.random.uniform(0.5, 2.5, 28)
    fraud_v = fraud_v * fraud_scale + fraud_shift
    
    # Combine
    V_data = np.vstack([normal_v, fraud_v])
    
    # Amounts
    normal_amounts = np.random.lognormal(3.5, 1.2, n_normal)
    fraud_amounts = np.random.lognormal(4.5, 1.5, n_fraud)
    amounts = np.hstack([normal_amounts, fraud_amounts])
    
    # Time
    times = np.random.uniform(0, 172800, n_samples)
    
    # Labels
    labels = np.hstack([np.zeros(n_normal), np.ones(n_fraud)])
    
    # Create DataFrame
    columns = [f'V{i}' for i in range(1, 29)]
    df = pd.DataFrame(V_data, columns=columns)
    df['Time'] = times
    df['Amount'] = amounts
    df['Class'] = labels.astype(int)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print("✅ Synthetic dataset generated!")

# Display basic info
print(f"\n📊 Dataset Shape: {df.shape}")
print(f"   Rows: {df.shape[0]:,}")
print(f"   Columns: {df.shape[1]}")

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Dataset info
df.info()

In [ ]:
# Statistical summary
df.describe()

## 2️⃣ Exploratory Data Analysis

In [ ]:
# Class distribution
class_counts = df['Class'].value_counts()
print("\n📊 Class Distribution:")
print(f"   Normal (0): {class_counts[0]:,} ({class_counts[0]/len(df)*100:.3f}%)")
print(f"   Fraud (1):  {class_counts[1]:,} ({class_counts[1]/len(df)*100:.3f}%)")
print(f"   Imbalance Ratio: 1:{int(class_counts[0]/class_counts[1])}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(data=df, x='Class', ax=ax1, palette=['#00d4ff', '#ff4444'])
ax1.set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Class (0=Normal, 1=Fraud)')
ax1.set_ylabel('Count')
ax1.set_yscale('log')  # Log scale to see fraud clearly

# Pie chart
ax2.pie(class_counts, labels=['Normal', 'Fraud'], autopct='%1.3f%%',
        colors=['#00d4ff', '#ff4444'], startangle=90)
ax2.set_title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Amount distribution by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df[df['Class']==0]['Amount'], bins=50, alpha=0.7, label='Normal', color='#00d4ff')
axes[0].hist(df[df['Class']==1]['Amount'], bins=50, alpha=0.7, label='Fraud', color='#ff4444')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Transaction Amount Distribution', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].set_yscale('log')

# Box plot
df.boxplot(column='Amount', by='Class', ax=axes[1])
axes[1].set_xlabel('Class (0=Normal, 1=Fraud)')
axes[1].set_ylabel('Amount ($)')
axes[1].set_title('Amount by Class', fontsize=14, fontweight='bold')
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.show()

print("\n📊 Amount Statistics:")
print(f"   Normal - Mean: ${df[df['Class']==0]['Amount'].mean():.2f}, Median: ${df[df['Class']==0]['Amount'].median():.2f}")
print(f"   Fraud  - Mean: ${df[df['Class']==1]['Amount'].mean():.2f}, Median: ${df[df['Class']==1]['Amount'].median():.2f}")

In [ ]:
# Time distribution
df['Hour'] = (df['Time'] / 3600) % 24

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time distribution by class
axes[0].hist(df[df['Class']==0]['Hour'], bins=24, alpha=0.7, label='Normal', color='#00d4ff')
axes[0].hist(df[df['Class']==1]['Hour'], bins=24, alpha=0.7, label='Fraud', color='#ff4444')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Count')
axes[0].set_title('Transaction Distribution by Hour', fontsize=14, fontweight='bold')
axes[0].legend()

# Fraud rate by hour
fraud_by_hour = df.groupby(df['Hour'].astype(int))['Class'].mean() * 100
axes[1].bar(fraud_by_hour.index, fraud_by_hour.values, color='#ff4444', alpha=0.7)
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Hour', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (sample of V features)
v_cols = ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'Amount', 'Class']
corr_matrix = df[v_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap (Selected Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3️⃣ Feature Engineering

In [ ]:
# Import feature engineering module
import sys
sys.path.append('..')
from src.feature_engineering import FeatureEngineer

# Initialize feature engineer
feature_engineer = FeatureEngineer()

# Create features
df_features = feature_engineer.create_features(df.copy())

print(f"\n✅ Feature engineering completed!")
print(f"   Original features: {df.shape[1]}")
print(f"   After engineering: {df_features.shape[1]}")
print(f"   New features: {df_features.shape[1] - df.shape[1]}")

In [ ]:
# Display new features
new_features = [col for col in df_features.columns if col not in df.columns]
print("\n🔧 Engineered Features:")
for i, feat in enumerate(new_features, 1):
    print(f"   {i}. {feat}")

## 4️⃣ Model Training

In [ ]:
# Prepare data
X = df_features.drop('Class', axis=1)
y = df_features['Class']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Data Split:")
print(f"   Training: {len(X_train):,} samples")
print(f"   Testing: {len(X_test):,} samples")
print(f"   Train fraud rate: {y_train.mean()*100:.3f}%")
print(f"   Test fraud rate: {y_test.mean()*100:.3f}%")

In [ ]:
# Handle imbalance with SMOTE
print("\n⚖️ Applying SMOTE...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"   Original: {len(X_train):,} samples")
print(f"   Resampled: {len(X_train_resampled):,} samples")
print(f"   New fraud rate: {y_train_resampled.mean()*100:.1f}%")

In [ ]:
# Train Random Forest
print("\n🌲 Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
rf_model.fit(X_train_resampled, y_train_resampled)
print("✅ Random Forest trained!")

In [ ]:
# Evaluate Random Forest
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print("\n📊 Random Forest Evaluation:")
print(f"\nROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Normal', 'Fraud']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance, y='feature', x='importance', palette='viridis')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title('Top 20 Feature Importances (Random Forest)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5️⃣ Prediction Examples

In [ ]:
# Get a random normal transaction
normal_sample = df_features[df_features['Class'] == 0].sample(1, random_state=42)
X_normal = normal_sample.drop('Class', axis=1)

# Predict
pred_proba_normal = rf_model.predict_proba(X_normal)[0][1]
pred_normal = rf_model.predict(X_normal)[0]

print("\n💳 Normal Transaction Example:")
print(f"   Amount: ${normal_sample['Amount'].values[0]:.2f}")
print(f"   Fraud Probability: {pred_proba_normal*100:.2f}%")
print(f"   Prediction: {'✅ NORMAL' if pred_normal == 0 else '⚠️ FRAUD'}")

In [ ]:
# Get a random fraud transaction
fraud_sample = df_features[df_features['Class'] == 1].sample(1, random_state=42)
X_fraud = fraud_sample.drop('Class', axis=1)

# Predict
pred_proba_fraud = rf_model.predict_proba(X_fraud)[0][1]
pred_fraud = rf_model.predict(X_fraud)[0]

print("\n🚨 Fraud Transaction Example:")
print(f"   Amount: ${fraud_sample['Amount'].values[0]:.2f}")
print(f"   Fraud Probability: {pred_proba_fraud*100:.2f}%")
print(f"   Prediction: {'✅ NORMAL' if pred_fraud == 0 else '⚠️ FRAUD'}")

## ✅ Conclusion

This notebook demonstrated:
- Data loading and exploration
- Class imbalance analysis
- Feature engineering
- SMOTE for handling imbalance
- Random Forest model training
- Model evaluation and interpretation

**Next Steps:**
1. Run the full training pipeline: `python train.py`
2. Launch the web app: `streamlit run app/app.py`
3. Experiment with XGBoost and ensemble methods
4. Try different imbalance handling techniques
5. Tune hyperparameters for better performance